# 🤖 Multi-Agent Demo: Smart Research Assistant

This notebook demonstrates multiple AI agents working together to answer complex queries.

## Architecture

```
                    ┌─────────────────┐
                    │   SUPERVISOR    │  ← Orchestrates & routes queries
                    │     AGENT       │
                    └────────┬────────┘
                             │
           ┌─────────────────┼─────────────────┐
           │                 │                 │
           ▼                 ▼                 ▼
    ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
    │  📚 RAG     │   │  🌍 WEB     │   │  🧮 MATH    │
    │  AGENT      │   │  AGENT      │   │  AGENT      │
    │             │   │             │   │             │
    │ Searches    │   │ Searches    │   │ Performs    │
    │ local docs  │   │ internet    │   │ calculations│
    └─────────────┘   └─────────────┘   └─────────────┘
```

**Agents:**
- 🎭 **Supervisor** - Routes queries to appropriate specialist
- 📚 **RAG Agent** - Searches local knowledge base (ChromaDB + Embeddings)
- 🌍 **Web Agent** - Searches the internet
- 🧮 **Math Agent** - Performs calculations & analysis

## Step 1: Install Dependencies

Run this cell if you haven't installed the required packages yet.

In [ ]:
# Uncomment and run if needed
# !pip install langchain langchain-openai langchain-community langgraph chromadb duckduckgo-search ddgs rich

## Step 2: Imports & Configuration

Import all necessary libraries and set up SSL handling.

In [ ]:
# ==============================================================================
# IMPORTS
# ==============================================================================

import os
import sys
from typing import TypedDict, Annotated, Sequence, Literal
from datetime import datetime
import operator

# SSL Fix for macOS / Corporate Networks
import ssl
import httpx
custom_http_client = httpx.Client(verify=False)

# Suppress SSL warnings for cleaner output
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# LangGraph for multi-agent orchestration
from langgraph.graph import StateGraph, END, START

# Vector store for RAG
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Web search (free, no API key needed!)
from langchain_community.tools import DuckDuckGoSearchRun

# For nice display in notebooks
from IPython.display import display, Markdown, HTML

print("✅ All imports successful!")

## Step 3: Set Your OpenAI API Key

⚠️ **Replace with your actual API key**

In [ ]:
# Set your OpenAI API key here
# Option 1: Set directly (not recommended for production)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

# Option 2: It's already set in your environment (recommended)
if not os.getenv("OPENAI_API_KEY"):
    print("⚠️ OPENAI_API_KEY not found! Please set it above.")
else:
    print("✅ OpenAI API key found!")

## Step 4: Initialize the LLM

We use GPT-4o-mini for cost efficiency. You can change to `gpt-4o` for better reasoning.

In [ ]:
# ==============================================================================
# INITIALIZE LLM
# ==============================================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",      # Cost-effective, fast model
    temperature=0,             # Low temperature for consistent outputs
    streaming=True,            # Enable streaming
    http_client=custom_http_client  # SSL fix
)

print("✅ LLM initialized: gpt-4o-mini")

## Step 5: Build the RAG Knowledge Base

This loads documents from the `knowledge_base/` folder, chunks them, and creates vector embeddings.

In [ ]:
# ==============================================================================
# BUILD RAG KNOWLEDGE BASE
# ==============================================================================

def build_knowledge_base():
    """Build vector store from local documents for RAG."""
    
    kb_path = "knowledge_base"  # Relative to notebook location
    
    if not os.path.exists(kb_path):
        print(f"⚠️ Knowledge base folder not found at {kb_path}")
        return None
    
    # Load all .txt files
    loader = DirectoryLoader(
        kb_path,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={'encoding': 'utf-8'}
    )
    documents = loader.load()
    print(f"📄 Loaded {len(documents)} documents")
    
    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    splits = text_splitter.split_documents(documents)
    print(f"📝 Created {len(splits)} text chunks")
    
    # Create embeddings and vector store
    embeddings = OpenAIEmbeddings(http_client=custom_http_client)
    
    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,
        collection_name="demo_kb"
    )
    
    print("✅ Knowledge base ready!")
    return vectorstore

# Build the knowledge base
vectorstore = build_knowledge_base()

## Step 6: Define Agent State

This is the shared "memory" that flows between all agents.

In [ ]:
# ==============================================================================
# DEFINE AGENT STATE (shared memory between agents)
# ==============================================================================

class AgentState(TypedDict):
    """State that flows through the multi-agent graph."""
    messages: Annotated[Sequence[BaseMessage], operator.add]
    query: str
    rag_response: str
    web_response: str
    math_response: str
    next_agent: str
    final_response: str

print("✅ AgentState defined")

## Step 7: Create the Agents

Each agent is a specialized function that handles specific types of queries.

In [ ]:
# ==============================================================================
# RAG AGENT - Searches local knowledge base
# ==============================================================================

def rag_agent(state: AgentState) -> dict:
    """RAG Agent: Searches local knowledge base."""
    
    query = state["query"]
    
    display(HTML(f'''
    <div style="padding: 10px; border: 2px solid #00CED1; border-radius: 10px; margin: 10px 0;">
        <h3>📚 RAG AGENT</h3>
        <p>🔍 Searching knowledge base for: <em>"{query}"</em></p>
    </div>
    '''))
    
    if vectorstore is None:
        response = "Knowledge base not available."
        return {"rag_response": response}
    
    # Search vector store
    docs = vectorstore.similarity_search(query, k=3)
    
    if not docs:
        response = "No relevant information found in knowledge base."
        return {"rag_response": response}
    
    print(f"  → Found {len(docs)} relevant chunks")
    
    # Combine context and generate response
    context = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Answer based on this context from our knowledge base. Be concise.
        
        CONTEXT:
        {context}"""),
        ("human", "{query}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"context": context, "query": query})
    
    display(HTML(f'''
    <div style="padding: 10px; background-color: #E0FFFF; border-radius: 10px; margin: 10px 0;">
        <strong>📚 RAG Response:</strong><br>{response}
    </div>
    '''))
    
    return {"rag_response": response}

print("✅ RAG Agent created")

In [ ]:
# ==============================================================================
# WEB AGENT - Searches the internet
# ==============================================================================

# Initialize DuckDuckGo search (free, no API key!)
search_tool = DuckDuckGoSearchRun()

def web_agent(state: AgentState) -> dict:
    """Web Agent: Searches the internet."""
    
    query = state["query"]
    
    display(HTML(f'''
    <div style="padding: 10px; border: 2px solid #32CD32; border-radius: 10px; margin: 10px 0;">
        <h3>🌍 WEB AGENT</h3>
        <p>🔍 Searching the web for: <em>"{query}"</em></p>
    </div>
    '''))
    
    try:
        # Perform web search
        search_results = search_tool.invoke(query)
        print(f"  → Retrieved web results")
        
        # Synthesize results with LLM
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Synthesize these search results into a clear answer.
            
            SEARCH RESULTS:
            {search_results}"""),
            ("human", "{query}")
        ])
        
        chain = prompt | llm | StrOutputParser()
        response = chain.invoke({"search_results": search_results, "query": query})
        
    except Exception as e:
        response = f"Web search error: {str(e)}"
    
    display(HTML(f'''
    <div style="padding: 10px; background-color: #98FB98; border-radius: 10px; margin: 10px 0;">
        <strong>🌍 Web Response:</strong><br>{response}
    </div>
    '''))
    
    return {"web_response": response}

print("✅ Web Agent created")

In [ ]:
# ==============================================================================
# MATH AGENT - Performs calculations
# ==============================================================================

def math_agent(state: AgentState) -> dict:
    """Math Agent: Performs calculations and analysis."""
    
    query = state["query"]
    
    display(HTML(f'''
    <div style="padding: 10px; border: 2px solid #FFD700; border-radius: 10px; margin: 10px 0;">
        <h3>🧮 MATH AGENT</h3>
        <p>🔢 Analyzing: <em>"{query}"</em></p>
    </div>
    '''))
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a math assistant. Solve step-by-step. Show your work."""),
        ("human", "{query}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke({"query": query})
    
    display(HTML(f'''
    <div style="padding: 10px; background-color: #FFFACD; border-radius: 10px; margin: 10px 0;">
        <strong>🧮 Math Response:</strong><br>{response}
    </div>
    '''))
    
    return {"math_response": response}

print("✅ Math Agent created")

In [ ]:
# ==============================================================================
# SUPERVISOR AGENT - Routes queries to appropriate agents
# ==============================================================================

def supervisor(state: AgentState) -> dict:
    """Supervisor: Routes queries to appropriate agents."""
    
    query = state["query"]
    
    display(HTML(f'''
    <div style="padding: 10px; border: 2px solid #9932CC; border-radius: 10px; margin: 10px 0;">
        <h3>🎭 SUPERVISOR AGENT</h3>
        <p>🎯 Analyzing query to determine best agent: <em>"{query}"</em></p>
    </div>
    '''))
    
    routing_prompt = ChatPromptTemplate.from_messages([
        ("system", """Route this query to the right agent.
        
        Available agents:
        - RAG: Company policies, internal docs, product info
        - WEB: Current events, real-time data, external info
        - MATH: Calculations, statistics, numerical analysis
        - ALL: Complex queries needing multiple sources
        
        Respond with ONLY: RAG, WEB, MATH, or ALL"""),
        ("human", "{query}")
    ])
    
    chain = routing_prompt | llm | StrOutputParser()
    decision = chain.invoke({"query": query}).strip().upper()
    
    # Clean up decision
    if "RAG" in decision:
        decision = "RAG"
    elif "WEB" in decision:
        decision = "WEB"
    elif "MATH" in decision:
        decision = "MATH"
    else:
        decision = "ALL"
    
    display(HTML(f'''
    <div style="padding: 10px; background-color: #DDA0DD; border-radius: 10px; margin: 10px 0;">
        <strong>📍 Routing Decision: {decision}</strong>
    </div>
    '''))
    
    return {"next_agent": decision}

print("✅ Supervisor Agent created")

In [ ]:
# ==============================================================================
# SYNTHESIZER - Combines responses into final answer
# ==============================================================================

def synthesizer(state: AgentState) -> dict:
    """Synthesize all agent responses into final answer."""
    
    display(HTML('''
    <div style="padding: 10px; border: 2px solid #4169E1; border-radius: 10px; margin: 10px 0;">
        <h3>🎯 SYNTHESIZER</h3>
        <p>📝 Combining responses from all agents...</p>
    </div>
    '''))
    
    responses = []
    
    if state.get("rag_response"):
        responses.append(f"**From Knowledge Base:**\n{state['rag_response']}")
    if state.get("web_response"):
        responses.append(f"**From Web Search:**\n{state['web_response']}")
    if state.get("math_response"):
        responses.append(f"**From Math Analysis:**\n{state['math_response']}")
    
    if not responses:
        final = "No information gathered from agents."
    elif len(responses) == 1:
        final = responses[0]
    else:
        combined = "\n\n".join(responses)
        prompt = ChatPromptTemplate.from_messages([
            ("system", """Combine these agent responses into one coherent answer.
            
            {responses}"""),
            ("human", "Synthesize for: {query}")
        ])
        chain = prompt | llm | StrOutputParser()
        final = chain.invoke({"responses": combined, "query": state["query"]})
    
    return {"final_response": final}

print("✅ Synthesizer created")

## Step 8: Build the Multi-Agent Graph

This connects all agents into a workflow using LangGraph.

In [ ]:
# ==============================================================================
# BUILD THE MULTI-AGENT GRAPH
# ==============================================================================

def build_graph():
    """Build the LangGraph multi-agent workflow."""
    
    # Create graph
    workflow = StateGraph(AgentState)
    
    # Add nodes (agents)
    workflow.add_node("supervisor", supervisor)
    workflow.add_node("rag_agent", rag_agent)
    workflow.add_node("web_agent", web_agent)
    workflow.add_node("math_agent", math_agent)
    workflow.add_node("synthesizer", synthesizer)
    
    # Routing logic
    def route_query(state: AgentState) -> str:
        decision = state.get("next_agent", "ALL")
        if decision == "RAG":
            return "rag_only"
        elif decision == "WEB":
            return "web_only"
        elif decision == "MATH":
            return "math_only"
        else:
            return "all_agents"
    
    def check_if_all(state: AgentState) -> str:
        if state.get("next_agent") == "ALL":
            if not state.get("web_response"):
                return "continue_to_web"
            elif not state.get("math_response"):
                return "continue_to_math"
        return "go_to_synthesizer"
    
    # Add edges
    workflow.add_edge(START, "supervisor")
    
    workflow.add_conditional_edges(
        "supervisor",
        route_query,
        {
            "rag_only": "rag_agent",
            "web_only": "web_agent",
            "math_only": "math_agent",
            "all_agents": "rag_agent"
        }
    )
    
    workflow.add_conditional_edges(
        "rag_agent",
        check_if_all,
        {"continue_to_web": "web_agent", "go_to_synthesizer": "synthesizer"}
    )
    
    workflow.add_conditional_edges(
        "web_agent",
        check_if_all,
        {"continue_to_math": "math_agent", "go_to_synthesizer": "synthesizer"}
    )
    
    workflow.add_edge("math_agent", "synthesizer")
    workflow.add_edge("synthesizer", END)
    
    return workflow.compile()

# Build the graph
graph = build_graph()
print("✅ Multi-Agent Graph built successfully!")

## Step 9: Visualize the Graph (Optional)

Let's see the actual graph structure!

In [ ]:
# Visualize the graph (requires graphviz)
try:
    from IPython.display import Image
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Graph visualization not available: {e}")
    print("\nGraph structure:")
    print("START → Supervisor → [RAG/WEB/MATH/ALL] → Synthesizer → END")

## Step 10: Run a Query!

Now let's test our multi-agent system with some queries.

In [ ]:
# ==============================================================================
# HELPER FUNCTION TO RUN QUERIES
# ==============================================================================

def run_query(query: str):
    """Execute a query through the multi-agent system."""
    
    display(HTML(f'''
    <div style="padding: 15px; border: 3px solid #1E90FF; border-radius: 15px; margin: 20px 0; background-color: #F0F8FF;">
        <h2>📨 Query</h2>
        <p style="font-size: 16px;"><strong>{query}</strong></p>
    </div>
    '''))
    
    # Initial state
    initial_state = {
        "messages": [HumanMessage(content=query)],
        "query": query,
        "rag_response": "",
        "web_response": "",
        "math_response": "",
        "next_agent": "",
        "final_response": ""
    }
    
    # Run the graph
    result = graph.invoke(initial_state)
    
    # Display final response
    display(HTML(f'''
    <div style="padding: 15px; border: 3px solid #228B22; border-radius: 15px; margin: 20px 0; background-color: #F0FFF0;">
        <h2>✨ Final Response</h2>
        <div style="font-size: 14px;">{result.get("final_response", "No response")}</div>
    </div>
    '''))
    
    return result

### Demo Query 1: RAG Agent (Knowledge Base)

This query will be routed to the **RAG Agent** to search our company knowledge base.

In [ ]:
# Query that routes to RAG Agent
result = run_query("What are TechCorp's core values and vacation policy?")

### Demo Query 2: Web Agent (Internet Search)

This query will be routed to the **Web Agent** to search the internet.

In [ ]:
# Query that routes to Web Agent
result = run_query("What is the latest news about artificial intelligence?")

### Demo Query 3: Math Agent (Calculations)

This query will be routed to the **Math Agent** for calculations.

In [ ]:
# Query that routes to Math Agent
result = run_query("Calculate the total revenue if we sell 150 units at $299 each with a 15% discount")

### Demo Query 4: All Agents (Complex Query)

This complex query will use **ALL agents** for a comprehensive answer.

In [ ]:
# Query that routes to ALL agents
result = run_query("Compare our company's AI product features with current market trends and calculate potential market share")

## 🎮 Try Your Own Queries!

Modify the query below and run the cell to test different scenarios.

In [ ]:
# Try your own query!
my_query = "What is our remote work policy?"  # Change this!

result = run_query(my_query)

---

## 📚 Summary

In this demo, we built a **multi-agent system** with:

1. **Supervisor Agent** - Intelligently routes queries to the right specialist
2. **RAG Agent** - Searches local knowledge base using vector embeddings
3. **Web Agent** - Searches the internet for current information
4. **Math Agent** - Handles calculations and numerical analysis
5. **Synthesizer** - Combines responses from multiple agents

### Key Technologies Used:
- **LangGraph** - Multi-agent orchestration
- **LangChain** - LLM and tool integration
- **ChromaDB** - Vector database for RAG
- **OpenAI GPT-4o-mini** - Language model
- **DuckDuckGo** - Free web search

### When to Use Multi-Agent Systems:
- Complex queries requiring multiple data sources
- Tasks needing specialized handling
- Workflows with routing decisions
- Systems that need to combine different capabilities